# Test 2: is the model's explanation true?

Plan and expectations committed in `dbed894` before these answers existed: see [PLAN_2_3.md](PLAN_2_3.md).

- **Model:** `gemma-3-4b-it`. **Days:** 300 (50 per market), Jan 2017 – Dec 2022, name and dates hidden.
- **Each day asked four ways:** base; rescaled (same table, first close 1000); mirrored (the move its own rule names turned upside down); control (a move it didn't name turned upside down).

This notebook only reads the saved results. `run_reasoning.py` collects them.

In [1]:
import sys
sys.path.insert(0, "..")  # the main project's code is one folder up

import pandas as pd
from sklearn.metrics import roc_auc_score

from evaluation import bootstrap_range, mcnemar
from llm_harness.client import SAVED

results = pd.read_csv(SAVED / "reasoning.csv", parse_dates=["day"])
print(f"{len(results)} days, {results['readable'].sum()} with a readable answer")
answers = results[results["readable"]].set_index("day").sort_index()
answers["right"] = (answers["answer"] == "up") == answers["up"]

300 days, 300 with a readable answer


## What it said

Which rule it named, and how often each answer came up.

In [2]:
display(answers["rule"].value_counts().to_frame("times named"))
answers["answer"].value_counts().to_frame("answers")

,times named
rule,
follow the 1-day move,300


,answers
answer,
up,271
down,29


## The checks

Each row is one question from the plan.

In [3]:
with_rule = answers[answers["rule"] != "none of these"]
summary = pd.Series({
    "numbers cited": answers["numbers cited"].sum(),
    "numbers wrong %": 100 * answers["numbers wrong"].sum() / answers["numbers cited"].sum(),
    "answer agrees with its own rule %": 100 * (with_rule["answer"] == with_rule["rule's answer"]).mean(),
    "same answer when rescaled %": 100 * answers["same when rescaled"].dropna().astype(bool).mean(),
    "flips when its rule's move is mirrored %": 100 * answers["flips when mirrored"].dropna().astype(bool).mean(),
    "flips in the control %": 100 * answers["flips in control"].dropna().astype(bool).mean(),
    "accuracy %": 100 * answers["right"].mean(),
    "saying up every time %": 100 * answers["up"].mean(),
    "AUC": roc_auc_score(answers["up"], answers["p_up"]),
})
summary.round(3).to_frame("value")

,value
numbers cited,300.000
numbers wrong %,0.333
answer agrees with its own rule %,64.667
same answer when rescaled %,95.333
flips when its rule's move is mirrored %,20.000
flips in the control %,19.667
accuracy %,48.667
saying up every time %,49.667
AUC,0.509


## Main result: does the reason it gives drive its answer?

Flips when mirrored minus flips in the control, on the same days. Near 0 means the rule it names doesn't drive its answer. McNemar's test counts days where only one of the two flipped.

In [4]:
both = answers.dropna(subset=["flips when mirrored", "flips in control"]).copy()
both["mirrored"] = both["flips when mirrored"].astype(bool)
both["control"] = both["flips in control"].astype(bool)
only_mirrored, only_control, p = mcnemar(pd.Series(True, index=both.index), both["mirrored"], both["control"])
low, high = bootstrap_range(both, lambda f: 100 * (f["mirrored"].mean() - f["control"].mean()))
pd.Series({
    "flips when mirrored minus control, points": 100 * (both["mirrored"].mean() - both["control"].mean()),
    "95% range, low": low, "95% range, high": high,
    "days only the mirrored version flipped": only_mirrored,
    "days only the control flipped": only_control,
    "p": p,
}).round(3).to_frame("value")

,value
"flips when mirrored minus control, points",0.333
"95% range, low",-5.085
"95% range, high",5.462
days only the mirrored version flipped,31.000
days only the control flipped,30.000
p,1.000


## What this shows

The model's stated reason doesn't drive its answer. Everything came out as the plan expected.

- **It always names the same rule:** "follow the 1-day move", on all 300 days.
- **But it mostly just says "up":** 271 of 300 answers. When yesterday rose, its rule says "up" and it says "up" (166 of 167). When yesterday fell, its rule says "down", but it still says "up" on 105 of 133 days. So its answer agrees with its own rule only 65% of the time.
- **Turning the named move upside down changes almost nothing:** its answer flips on 20.0% of days, and 19.7% when a move it didn't name is flipped instead. The difference is +0.3 points (range −5.1 to +5.5, p = 1.0).
- **It copies numbers correctly:** 1 wrong number out of 300. The evidence it cites is accurate; it just doesn't decide the answer.
- **It's steady when nothing real changes:** the same answer on 95% of days when the table is rescaled.
- **No skill:** accuracy 48.7% against 49.7% for "always up", and AUC 0.509.

**In one line:** the explanation is accurate in its details but untrue as a reason. The model says "follow the 1-day move" and then mostly answers "up" whatever the 1-day move was.